# English to Hinglish Machine Translation

This notebook demonstrates how to load, preprocess, train, and evaluate an LSTM-based sequence-to-sequence model to translate English sentences into Hinglish.

In [ ]:
import kagglehub

# Download latest version of the dataset
path = kagglehub.dataset_download("yash9439/eng-hinglish-machine-translation")
print("Path to dataset files:", path)

Path to dataset files: /root/.cache/kagglehub/datasets/yash9439/eng-hinglish-machine-translation/versions/1


In [ ]:
import os
import pandas as pd
import re

# List the files to find the CSV or text dataset
files = os.listdir(path)
print("Files in dataset directory:", files)

# Assuming there is a CSV file in the directory, let's find and load it
csv_files = [f for f in files if f.endswith('.csv')]
if csv_files:
    df_path = os.path.join(path, csv_files[0])
    df = pd.read_csv(df_path)
    print(f"Loaded dataset with {len(df)} rows.")
    display(df.head())
else:
    print("No CSV file found, listing all files:", files)

Files in dataset directory: ['Eng - Hinglish']
No CSV file found, listing all files: ['Eng - Hinglish']


In [7]:
import os
import pandas as pd

# Let's check the contents of the 'Eng - Hinglish' folder
sub_dir = os.path.join(path, 'Eng - Hinglish')
if os.path.exists(sub_dir):
    sub_files = os.listdir(sub_dir)
    print("Files inside sub-directory:", sub_files)

    # Try to find a CSV or text file to load
    target_file = [f for f in sub_files if f.endswith('.csv') or f.endswith('.txt')]
    if target_file:
        df_path = os.path.join(sub_dir, target_file[0])
        # Let's inspect the first few lines or read as CSV
        try:
            df = pd.read_csv(df_path)
        except Exception:
            df = pd.read_csv(df_path, sep='\t', names=['english', 'hinglish'])
        print(f"Successfully loaded dataset with {len(df)} rows.")
        display(df.head())
    else:
        print("No matching data files found in the sub-directory.")
else:
    print("Sub-directory does not exist.")

Files inside sub-directory: ['test.txt', 'train.txt', 'dev.txt']
Successfully loaded dataset with 942 rows.


,english,hinglish
0,hello,hello
1,"hello there , I have not seen this movie so im...","हैलो यार , मैं इस movie को नहीं देख हों तो , त..."
2,Alright that is fine . What is the movie ?,अच्छा तो इस movie किस बारे में हैं ?
3,The movie is The Social Network,is movie tho social network के बारे में हैं
4,I have not seen that one either .,मैं ऐसे कुछ नहीं देखा हूँ


Error: Runtime no longer has a reference to this dataframe, please re-run this cell and try again.
Error: Runtime no longer has a reference to this dataframe, please re-run this cell and try again.
Error: Runtime no longer has a reference to this dataframe, please re-run this cell and try again.
Error: Runtime no longer has a reference to this dataframe, please re-run this cell and try again.


In [ ]:
def preprocess_sentence(sentence):
    if not isinstance(sentence, str):
        return ""
    # Lowercase, clean up special characters, and strip spacing
    sentence = sentence.lower().strip()
    sentence = re.sub(r"([^a-zA-Z0-9\s])", r" \1 ", sentence)
    sentence = re.sub(r"\s+", " ", sentence)
    return sentence.strip()

# We assume columns are named 'english' and 'hinglish' (or similar based on standard datasets)
# Let's clean the columns dynamically based on what's available
eng_col = [col for col in df.columns if 'eng' in col.lower()][0]
hing_col = [col for col in df.columns if 'hing' in col.lower() or 'hin' in col.lower()][0]

print(f"Using '{eng_col}' as English and '{hing_col}' as Hinglish column.")

df['cleaned_english'] = df[eng_col].apply(preprocess_sentence)
# For Hinglish translation targets, we add starting and ending tags for sequence-to-sequence learning
df['cleaned_hinglish'] = df[hing_col].apply(lambda x: f"<start> {preprocess_sentence(x)} <end>")

print("Sample preprocessed pairs:")
display(df[['cleaned_english', 'cleaned_hinglish']].head(5))

In [8]:
import os
import re
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split

# 1. Load all splits (train, dev, test) and combine them
all_dfs = []
for file_name in ['train.txt', 'dev.txt', 'test.txt']:
    file_path = os.path.join(sub_dir, file_name)
    if os.path.exists(file_path):
        try:
            # Try standard tab separated reading first
            temp_df = pd.read_csv(file_path, sep='\t', names=['english', 'hinglish'])
            all_dfs.append(temp_df)
        except Exception as e:
            print(f"Error reading {file_name}: {e}")

df = pd.concat(all_dfs, ignore_index=True) if all_dfs else df
print(f"Total dataset size combined: {len(df)} pairs.")

# Helper for clean preprocessing
def preprocess_sentence(sentence):
    if not isinstance(sentence, str):
        return ""
    sentence = sentence.lower().strip()
    # Keep letters, numbers, standard punctuation, and hindi characters
    sentence = re.sub(r"([^a-zA-Z0-9\s\u0900-\u097F])", r" \1 ", sentence)
    sentence = re.sub(r"\s+", " ", sentence)
    return sentence.strip()

# Apply cleaning
df['cleaned_english'] = df['english'].apply(preprocess_sentence)
df['cleaned_hinglish_in'] = df['hinglish'].apply(lambda x: f"<start> {preprocess_sentence(x)}")
df['cleaned_hinglish_out'] = df['hinglish'].apply(lambda x: f"{preprocess_sentence(x)} <end>")

print("Example:")
print("Input:", df['cleaned_english'].iloc[1])
print("Decoder Input:", df['cleaned_hinglish_in'].iloc[1])
print("Decoder Target:", df['cleaned_hinglish_out'].iloc[1])

Total dataset size combined: 9002 pairs.
Example:
Input: the director is zack snyder , 27 % rotten tomatoes , 4 . 9 / 10 .
Decoder Input: <start> zack snyder director हैं , 27 % rotten tomatoes , 4 . 9 / 10 .
Decoder Target: zack snyder director हैं , 27 % rotten tomatoes , 4 . 9 / 10 . <end>


In [9]:
# 2. Tokenize and Build Vocabularies
# We will use distinct tokenizers for English (Source) and Hinglish (Target)
src_tokenizer = Tokenizer(filters='')
src_tokenizer.fit_on_texts(df['cleaned_english'])
src_sequences = src_tokenizer.texts_to_sequences(df['cleaned_english'])

targ_tokenizer = Tokenizer(filters='')
# Fit on both decoder inputs and targets to cover '<start>' and '<end>'
targ_tokenizer.fit_on_texts(df['cleaned_hinglish_in'].tolist() + df['cleaned_hinglish_out'].tolist())
targ_sequences_in = targ_tokenizer.texts_to_sequences(df['cleaned_hinglish_in'])
targ_sequences_out = targ_tokenizer.texts_to_sequences(df['cleaned_hinglish_out'])

# Determine maximum sequence lengths
max_len_src = max(len(s) for s in src_sequences)
max_len_targ = max(len(s) for s in targ_sequences_in)

# Padding
encoder_input_data = pad_sequences(src_sequences, maxlen=max_len_src, padding='post')
decoder_input_data = pad_sequences(targ_sequences_in, maxlen=max_len_targ, padding='post')
decoder_target_data = pad_sequences(targ_sequences_out, maxlen=max_len_targ, padding='post')

# Vocab sizes (add 1 for 0-padding)
num_encoder_tokens = len(src_tokenizer.word_index) + 1
num_decoder_tokens = len(targ_tokenizer.word_index) + 1

print(f"Encoder input vocabulary size: {num_encoder_tokens}")
print(f"Decoder input vocabulary size: {num_decoder_tokens}")
print(f"Max source sequence length: {max_len_src}")
print(f"Max target sequence length: {max_len_targ}")

Encoder input vocabulary size: 6104
Decoder input vocabulary size: 8582
Max source sequence length: 297
Max target sequence length: 378


In [10]:
# 3. Design the Seq2Seq LSTM network with Keras Functional API
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense
from tensorflow.keras.models import Model

latent_dim = 256
embedding_dim = 128

# Encoder
encoder_inputs = Input(shape=(None,))
enc_emb = Embedding(num_encoder_tokens, embedding_dim, mask_zero=True)(encoder_inputs)
encoder_outputs, state_h, state_c = LSTM(latent_dim, return_state=True)(enc_emb)
encoder_states = [state_h, state_c]

# Decoder
decoder_inputs = Input(shape=(None,))
dec_emb_layer = Embedding(num_decoder_tokens, embedding_dim, mask_zero=True)
dec_emb = dec_emb_layer(decoder_inputs)

dec_lstm = LSTM(latent_dim, return_sequences=True, return_state=True)
decoder_outputs, _, _ = dec_lstm(dec_emb, initial_state=encoder_states)

dec_dense = Dense(num_decoder_tokens, activation='softmax')
decoder_outputs = dec_dense(decoder_outputs)

# Define the full model
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, None, 128) │    781,312 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, None)      │          0 │ input_layer[0][0] │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, None, 128) │  1,098,496 │ input_layer_1[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ [(None, 256),     │    394,240 │ embedding[0][0],  │
│                     │ (None, 256),      │            │ not_equal[0][0]   │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ [(None, None,     │    394,240 │ embedding_1[0][0… │
│                     │ 256), (None,      │            │ lstm[0][1],       │
│                     │ 256), (None,      │            │ lstm[0][2]        │
│                     │ 256)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, None,      │  2,205,574 │ lstm_1[0][0]      │
│                     │ 8582)             │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 4,873,862 (18.59 MB)

 Trainable params: 4,873,862 (18.59 MB)

 Non-trainable params: 0 (0.00 B)

In [11]:
# 4. Train the Model
# We will do a simple split to have an evaluation dataset
indices = np.arange(encoder_input_data.shape[0])
np.random.shuffle(indices)

train_idx, val_idx = train_test_split(indices, test_size=0.1, random_state=42)

encoder_input_train = encoder_input_data[train_idx]
decoder_input_train = decoder_input_data[train_idx]
decoder_target_train = decoder_target_data[train_idx]

encoder_input_val = encoder_input_data[val_idx]
decoder_input_val = decoder_input_data[val_idx]
decoder_target_val = decoder_target_data[val_idx]

history = model.fit(
    [encoder_input_train, decoder_input_train],
    decoder_target_train,
    batch_size=32,
    epochs=15,
    validation_data=([encoder_input_val, decoder_input_val], decoder_target_val)
)

Epoch 1/15
254/254 ━━━━━━━━━━━━━━━━━━━━ 53s 170ms/step - accuracy: 0.0957 - loss: 6.2856 - val_accuracy: 0.1164 - val_loss: 5.8645
Epoch 2/15
254/254 ━━━━━━━━━━━━━━━━━━━━ 74s 170ms/step - accuracy: 0.1314 - loss: 5.5780 - val_accuracy: 0.1423 - val_loss: 5.5785
Epoch 3/15
254/254 ━━━━━━━━━━━━━━━━━━━━ 44s 173ms/step - accuracy: 0.1591 - loss: 5.2708 - val_accuracy: 0.1742 - val_loss: 5.4191
Epoch 4/15
254/254 ━━━━━━━━━━━━━━━━━━━━ 44s 172ms/step - accuracy: 0.1883 - loss: 5.0109 - val_accuracy: 0.1940 - val_loss: 5.2613
Epoch 5/15
254/254 ━━━━━━━━━━━━━━━━━━━━ 44s 173ms/step - accuracy: 0.2130 - loss: 4.7618 - val_accuracy: 0.2079 - val_loss: 5.1509
Epoch 6/15
254/254 ━━━━━━━━━━━━━━━━━━━━ 82s 173ms/step - accuracy: 0.2338 - loss: 4.5407 - val_accuracy: 0.2253 - val_loss: 5.0656
Epoch 7/15
254/254 ━━━━━━━━━━━━━━━━━━━━ 44s 172ms/step - accuracy: 0.2531 - loss: 4.3325 - val_accuracy: 0.2390 - val_loss: 4.9984
Epoch 8/15
254/254 ━━━━━━━━━━━━━━━━━━━━ 44s 173ms/step - accuracy: 0.2708 - loss: 4

In [12]:
# 5. Inference and Evaluation with BLEU score
from nltk.translate.bleu_score import corpus_bleu

# Set up inference models
encoder_model = Model(encoder_inputs, encoder_states)

decoder_state_input_h = Input(shape=(latent_dim,))
decoder_state_input_c = Input(shape=(latent_dim,))
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]

dec_emb_infer = dec_emb_layer(decoder_inputs)
dec_outputs_infer, state_h_infer, state_c_infer = dec_lstm(dec_emb_infer, initial_state=decoder_states_inputs)
dec_states_infer = [state_h_infer, state_c_infer]
dec_outputs_infer = dec_dense(dec_outputs_infer)

decoder_model = Model(
    [decoder_inputs] + decoder_states_inputs,
    [dec_outputs_infer] + dec_states_infer
)

# Reconstruct mapping from token index to words
target_index_to_word = {v: k for k, v in targ_tokenizer.word_index.items()}

def decode_sequence(input_seq):
    states_value = encoder_model.predict(input_seq, verbose=0)
    target_seq = np.zeros((1, 1))
    target_seq[0, 0] = targ_tokenizer.word_index['<start>']

    stop_condition = False
    decoded_sentence = []
    while not stop_condition:
        output_tokens, h, c = decoder_model.predict([target_seq] + states_value, verbose=0)
        sampled_token_index = np.argmax(output_tokens[0, -1, :])
        sampled_char = target_index_to_word.get(sampled_token_index, '')

        if sampled_char == '<end>' or len(decoded_sentence) > max_len_targ:
            stop_condition = True
        else:
            if sampled_char != '':
                decoded_sentence.append(sampled_char)

        target_seq = np.zeros((1, 1))
        target_seq[0, 0] = sampled_token_index
        states_value = [h, c]

    return decoded_sentence

# Evaluate some samples from Validation Data and calculate BLEU score
references = []
hypotheses = []

# Calculate for the first 30 validation sequences to keep evaluation swift
for i in range(min(30, len(encoder_input_val))):
    input_seq = encoder_input_val[i:i+1]
    decoded_words = decode_sequence(input_seq)

    # Target references
    ref_seq = decoder_target_val[i]
    actual_words = [target_index_to_word[idx] for idx in ref_seq if idx > 0 and target_index_to_word.get(idx, '') not in ['', '<end>']]

    references.append([actual_words])
    hypotheses.append(decoded_words)

# Print translation examples
for i in range(5):
    print(f"Source: {df['cleaned_english'].iloc[val_idx[i]]}")
    print(f"Target Reference: {' '.join(references[i][0])}")
    print(f"Predicted Translation: {' '.join(hypotheses[i])}")
    print("-" * 50)

score = corpus_bleu(references, hypotheses)
print(f"BLEU-1 Score on validation sample: {score:.4f}")

Source: i just re - watched back to the future
Target Reference: मैं भविष्य को एक बार देखा
Predicted Translation: मैं भी . मुझे लगता है कि यह एक अच्छा है .
--------------------------------------------------
Source: mine is blank also
Target Reference: मेरा भी blank हैं
Predicted Translation: मैं भी
--------------------------------------------------
Source: fairly old
Target Reference: fairly old
Predicted Translation: वह
--------------------------------------------------
Source: huh why is gibson pretending to be a british army private
Target Reference: huh gibson british army private होना क्यों pretend करार है
Predicted Translation: the imitation game में तो एक और villain के लिए
--------------------------------------------------
Source: i loved that hulk saved iron man when he was falling , such a great moment .
Target Reference: मुझे हलक पसंद है उसने iron man को save किया था जब वह गिर रहा था .
Predicted Translation: मुझे यह पसंद है कि वह एक अच्छा लगा कि वह एक और elsas का एक अच्छा मिश

/usr/local/lib/python3.13/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)


In [13]:
# Define 5 unseen English sentences
unseen_sentences = [
    "hello , how are you ?",
    "i like to watch action movies .",
    "can you tell me about the social network ?",
    "this movie is very good and interesting .",
    "i will watch back to the future tomorrow ."
]

print("=== Translating Unseen Sentences ===\n")
for sentence in unseen_sentences:
    # 1. Preprocess the unseen sentence
    cleaned = preprocess_sentence(sentence)

    # 2. Convert to sequences using the fitted source tokenizer
    seq = src_tokenizer.texts_to_sequences([cleaned])

    # 3. Pad sequence
    padded_seq = pad_sequences(seq, maxlen=max_len_src, padding='post')

    # 4. Decode translation using the inference models
    translated_words = decode_sequence(padded_seq)
    translation = " ".join(translated_words)

    print(f"English: {sentence}")
    print(f"Hinglish: {translation}")
    print("-" * 50)

=== Translating Unseen Sentences ===

English: hello , how are you ?
Hinglish: hello , कैसे हो ?
--------------------------------------------------
English: i like to watch action movies .
Hinglish: मुझे इस movie के बारे में है
--------------------------------------------------
English: can you tell me about the social network ?
Hinglish: क्या आप को इस movie को देख सकते हो ?
--------------------------------------------------
English: this movie is very good and interesting .
Hinglish: यह movie काफी funny है .
--------------------------------------------------
English: i will watch back to the future tomorrow .
Hinglish: मैं भी मानता हूँ .
--------------------------------------------------


In [14]:
new_unseen_sentences = [
    "i love this song .",
    "where is the theatre ?",
    "we are going to have fun today .",
    "did you see the new trailer ?",
    "please help me with this ."
]

print("=== Translating Another Set of Unseen Sentences ===\n")
for sentence in new_unseen_sentences:
    cleaned = preprocess_sentence(sentence)
    seq = src_tokenizer.texts_to_sequences([cleaned])
    padded_seq = pad_sequences(seq, maxlen=max_len_src, padding='post')
    translated_words = decode_sequence(padded_seq)
    translation = " ".join(translated_words)

    print(f"English: {sentence}")
    print(f"Hinglish: {translation}")
    print("-" * 50)

=== Translating Another Set of Unseen Sentences ===

English: i love this song .
Hinglish: मुझे याद है .
--------------------------------------------------
English: where is the theatre ?
Hinglish: क्या यह क्या है ?
--------------------------------------------------
English: we are going to have fun today .
Hinglish: आप के बारे में बात कर रहे हैं
--------------------------------------------------
English: did you see the new trailer ?
Hinglish: क्या तुमने climax देखी है ?
--------------------------------------------------
English: please help me with this .
Hinglish: sure , मुझे लगता है कि यह एक अच्छा है
--------------------------------------------------


In [15]:
# Interactive Translator
user_input = input("Enter an English sentence to translate to Hinglish: ")
if user_input.strip():
    cleaned = preprocess_sentence(user_input)
    seq = src_tokenizer.texts_to_sequences([cleaned])
    padded_seq = pad_sequences(seq, maxlen=max_len_src, padding='post')
    translated_words = decode_sequence(padded_seq)
    translation = " ".join(translated_words)

    print("\n--- Translation Result ---")
    print(f"English: {user_input}")
    print(f"Hinglish: {translation}")
else:
    print("Please enter a valid sentence.")

Enter an English sentence to translate to Hinglish: hello,can you tell me your name, sir

--- Translation Result ---
English: hello,can you tell me your name, sir
Hinglish: hello , आप को इस movie को पसंद हैं
